In [6]:
import pandas as pd
import mysql.connector
from mysql.connector import Error
import os
from dotenv import load_dotenv

# .env 파일 로드
load_dotenv()

# MySQL 설정
DB_HOST = os.getenv("DB_HOST")
DB_PORT = int(os.getenv("DB_PORT", 3306))
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_NAME = os.getenv("DB_NAME")
SERVICE_KEY = os.getenv("SERVICE_KEY")

# 연결 객체
conn = None
# MySQL 연결
def create_connection():
    """MySQL 연결을 생성하고 반환."""
    try:
        return mysql.connector.connect(
            host=DB_HOST,
            port=DB_PORT,
            user=DB_USER,
            password=DB_PASSWORD,
            database=DB_NAME,
            pool_name="mypool",
            pool_size=5
        )
    except mysql.connector.Error as e:
         return None


conn = None

def ensure_connection():
    global conn
    if conn is None or not conn.is_connected():
        print("Database connection is not active. Attempting to reconnect...")
        conn = create_connection()
        if conn and conn.is_connected():
            print("Reconnected to the database.")
        else:
            print("Failed to reconnect to the database.")

import pandas as pd

# MySQL 연결 확인 및 데이터 읽기
ensure_connection()

if conn and conn.is_connected():
    df1 = pd.read_sql("SELECT * FROM oceandata;", conn)
    df2 = pd.read_sql("SELECT * FROM aisdata;", conn)

    # CSV 저장
    df1.to_csv('../document/oceandata.csv', index=False, encoding='utf-8-sig')
    print("데이터가 'oceandata.csv'로 저장되었습니다.")
    df2.to_csv('../document/aisdata.csv', index=False, encoding='utf-8-sig')
    print("데이터가 'aisdata.csv'로 저장되었습니다.")
else:
    print("Failed to connect to the database.")


Database connection is not active. Attempting to reconnect...
Reconnected to the database.


C:\Users\user\AppData\Local\Temp\ipykernel_11800\1613750681.py:55: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df1 = pd.read_sql("SELECT * FROM oceandata;", conn)
C:\Users\user\AppData\Local\Temp\ipykernel_11800\1613750681.py:56: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df2 = pd.read_sql("SELECT * FROM aisdata;", conn)


데이터가 'oceandata.csv'로 저장되었습니다.


In [4]:
import pandas as pd
from datetime import timedelta

# CSV 파일 읽기
ais_data = pd.read_csv('../document/aisdata.csv')
ocean_data = pd.read_csv('../document/oceandata.csv')

# 시간 데이터 변환 (datetime 형식으로 변환)
ais_data['timestamp'] = pd.to_datetime(ais_data['timestamp'])
ocean_data['datetime'] = pd.to_datetime(ocean_data['datetime'])

# 병합을 위한 시간 차이 기준 (예: 10분)
time_threshold = timedelta(minutes=10)

# 병합 로직: 가장 가까운 datetime 매칭
def find_closest_with_threshold(row, ocean_df):
    time_differences = (ocean_df['datetime'] - row['timestamp']).abs()
    closest_idx = time_differences.idxmin()
    # 시간 차이가 허용 범위 내에 있는 경우에만 반환
    if time_differences.iloc[closest_idx] <= time_threshold:
        return ocean_df.loc[closest_idx]
    return pd.Series(index=ocean_df.columns)  # 허용 범위 초과 시 빈 데이터 반환

# 병합된 결과 저장
merged_data = ais_data.apply(
    lambda row: pd.concat([row, find_closest_with_threshold(row, ocean_data)]),
    axis=1
)

# 결과 저장
merged_data.to_csv('../document/merged_data.csv', index=False, encoding='utf-8-sig')
print("병합된 데이터가 'merged_data.csv'에 저장되었습니다.")


병합된 데이터가 'merged_data.csv'에 저장되었습니다.
